# sparrow-rockfinch: a C++ extension module for Python

[`rockfinch_demo`](https://github.com/alex-placet/compute2026-sparrow-presentation/tree/main/rockfinch-demo) is a Python extension module written in C++ with [nanobind](https://github.com/wjakob/nanobind).
It links sparrow, sparrow-compute and sparrow-rockfinch, is compiled to WebAssembly, and the Python kernel imports it like any compiled package.
Its full source is [`rockfinch_demo.cpp`](rockfinch_demo.cpp), next to this notebook in the file browser; abridged:

```c++
using sparrow::rockfinch::SparrowArray;

// Python -> C++: any object implementing __arrow_c_array__ (pyarrow.Array, SparrowArray, ...)
SparrowArray import_array(nb::handle data)
{
    nb::tuple capsules = nb::cast<nb::tuple>(data.attr("__arrow_c_array__")());
    return SparrowArray(capsules[0].ptr(), capsules[1].ptr());
}

// C++ -> Python: SparrowArray implements __arrow_c_array__
nb::class_<SparrowArray>(m, "SparrowArray")
    .def("__arrow_c_array__", [](const SparrowArray& self, nb::handle requested_schema)
    {
        auto [schema, array] = self.export_to_capsules();
        return nb::make_tuple(nb::steal(schema), nb::steal(array));
    }, "requested_schema"_a = nb::none());

m.def("hypot", [](nb::handle x, nb::handle y)
{
    const SparrowArray x_array = import_array(x);
    const SparrowArray y_array = import_array(y);
    const sp::primitive_array<double>& a = as_float64(x_array);
    const sp::primitive_array<double>& b = as_float64(y_array);
    auto result = sp::compute::eval(sp::compute::sqrt(a * a + b * b));
    return SparrowArray(sp::array(std::move(result)));
});
```

The demo notebooks: [sparrow.ipynb](sparrow.ipynb) (arrays), [sparrow-ipc.ipynb](sparrow-ipc.ipynb) (Arrow IPC with pyarrow), [sparrow-compute.ipynb](sparrow-compute.ipynb) (lazy compute) and [rockfinch.ipynb](rockfinch.ipynb) (C++ extension module for Python).

In [ ]:
import pyarrow as pa
import polars as pl
import rockfinch_demo

print("pyarrow", pa.__version__, "| polars", pl.__version__)
print("sparrow", rockfinch_demo.sparrow_version, "| sparrow-rockfinch", rockfinch_demo.rockfinch_version)

### From C++ to Python

An array built in C++ is consumed by Polars and pyarrow through `__arrow_c_array__`.

In [ ]:
cpp_array = rockfinch_demo.make_array()
print(cpp_array)
print(pl.Series(cpp_array))
print(pa.array(cpp_array))

### From Python to C++ and back

Two Polars columns go to C++, sparrow-compute computes the distance to the origin, and the result comes back as a Polars column.

In [ ]:
points = pl.DataFrame({"x": [3.0, 5.0, 8.0, 20.0], "y": [4.0, 12.0, 15.0, 21.0]})
distance = rockfinch_demo.hypot(pa.array(points["x"]), pa.array(points["y"]))
print(points.with_columns(distance=pl.Series(distance)))

### Under the hood: the Arrow PyCapsule Interface

`__arrow_c_array__` returns an `ArrowSchema` and an `ArrowArray` capsule (Arrow C Data Interface).
`SparrowArray(schema_capsule, array_capsule)` hands them to `sparrow::rockfinch::import_array_from_capsules`,
which takes ownership of the C structures; the buffers themselves are not copied.

In [ ]:
arrow_array = pa.array([1, 2, None, 4, 5], type=pa.int32())
schema_capsule, array_capsule = arrow_array.__arrow_c_array__()
print(schema_capsule)
print(array_capsule)

In [ ]:
sparrow_array = rockfinch_demo.SparrowArray(arrow_array)
print(sparrow_array, len(sparrow_array))

The Polars snippet from the slides:

In [ ]:
from polars._plr import PySeries
from polars._utils.wrap import wrap_s

ps = PySeries.from_arrow_c_array(sparrow_array)
print(wrap_s(ps))